# FixFirst AI Colab GPU Training

Use this notebook in Google Colab, not in the local VS Code kernel. It installs the training dependencies, mounts Google Drive, and runs the sentiment training pipeline on Colab's free GPU.

Note: the repo currently has the full sentiment training path wired up. The aspect-category trainer is not implemented in this workspace yet, so this notebook focuses on the sentiment training flow.

In [2]:
from google.colab import auth

auth.authenticate_user()

## 2. Authenticate with Google Colab

If you want to persist checkpoints and artifacts, authenticate with Google and use Drive as the storage layer.

In [3]:
import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
import torch

print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
CUDA available: False


In [4]:
# Install dependencies for the training run
!pip -q install --upgrade pip
!pip -q install -r /content/FixFirst_AI/requirements.txt
!pip -q install -r /content/FixFirst_AI/requirements-training.txt
!pip -q install -e /content/FixFirst_AI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.0 MB/s eta 0:00:0000:0100:01
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/FixFirst_AI/requirements.txt'
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/FixFirst_AI/requirements-training.txt'
ERROR: /content/FixFirst_AI is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).


## 1. Install and Import Required Libraries

Colab does not use the local VS Code kernel. Run this notebook inside Colab so the runtime can attach to a free GPU.

In [5]:
# If you keep the repo in Drive, sync it into the Colab filesystem.
if DRIVE_ROOT.exists():
    !rm -rf /content/FixFirst_AI
    !cp -r /content/drive/MyDrive/FixFirst_AI /content/FixFirst_AI

os.chdir('/content/FixFirst_AI')
print('Working directory:', Path.cwd())
print('Train parquet exists:', Path('data/processed/train.parquet').exists())
print('Silver labels exist:', Path('data/silver_labels/silver_labels.parquet').exists())

NameError: name 'DRIVE_ROOT' is not defined

In [ ]:
def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    predictions = logits.argmax(axis=-1)
    accuracy = (predictions == true_labels).mean().item()
    return {'accuracy': accuracy}

training_args = TrainingArguments(
    output_dir='/content/fixfirst_aspect_sentiment_checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=25,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

In [ ]:
artifact_dir = DRIVE_ROOT / 'artifacts' / 'models' / 'aspect_sentiment' / 'colab_final'
artifact_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(artifact_dir))
tokenizer.save_pretrained(str(artifact_dir))

metrics_path = artifact_dir / 'metrics.json'
metrics_path.write_text(pd.Series(metrics).to_json(indent=2))

print('Saved model to:', artifact_dir)
print('Saved metrics to:', metrics_path)

## 8. Save and Download Results

Save the trained model, tokenizer, and training metrics to Google Drive so you can reuse them outside Colab.

## 7. Train the Model on GPU

Use a small smoke test first, then run the full training loop once the notebook is working.

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback

from fixfirst.config.settings import settings
from fixfirst.models.aspect_sentiment.dataset import AspectSentimentDataset, SENTIMENT_LABELS, build_sentiment_examples
from fixfirst.labeling.taxonomy import load_active_taxonomy

silver_path = Path('data/silver_labels/silver_labels.parquet')
if not silver_path.exists():
    silver_path = DRIVE_ROOT / 'data' / 'silver_labels' / 'silver_labels.parquet'

labels_df = pd.read_parquet(silver_path)
taxonomy = load_active_taxonomy()
feature_display_names = {t['feature_key']: t['display_name'] for t in taxonomy}
examples_df = build_sentiment_examples(labels_df, feature_display_names)

print('Training examples:', len(examples_df))
print(examples_df.head())

BASE_MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
VAL_SIZE = 0.1
RANDOM_STATE = 42

label_counts = examples_df['label'].value_counts()
can_stratify = (label_counts >= 2).all() and examples_df['label'].nunique() > 1
train_df, val_df = train_test_split(
    examples_df,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=examples_df['label'] if can_stratify else None,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(SENTIMENT_LABELS),
)

train_dataset = AspectSentimentDataset(
    train_df['text_a'].tolist(),
    train_df['text_b'].tolist(),
    train_df['label'].tolist(),
    tokenizer,
    max_length=MAX_LENGTH,
)
val_dataset = AspectSentimentDataset(
    val_df['text_a'].tolist(),
    val_df['text_b'].tolist(),
    val_df['label'].tolist(),
    tokenizer,
    max_length=MAX_LENGTH,
)

print('train split:', len(train_df), 'val split:', len(val_df))

## 6. Build and Compile the Model

This notebook fine-tunes a transformer classifier on the silver labels stored in Drive.

## 5. Upload or Load Training Data

Copy the repo to the Colab VM and point the pipeline at data stored in Drive so checkpoints survive disconnects.

In [ ]:
!nvidia-smi
print('torch cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device count:', torch.cuda.device_count())
    print('current device:', torch.cuda.current_device())
    print('device name:', torch.cuda.get_device_name(0))

## 4. Configure GPU Runtime

Colab free tier can provide a T4 GPU when available. Confirm the runtime sees CUDA before starting training.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/FixFirst_AI')
REPO_ROOT = Path('/content/FixFirst_AI')
print('Drive root:', DRIVE_ROOT)
print('Repo root:', REPO_ROOT)

## 3. Mount Google Drive

Use Drive to persist the repository, dataset, checkpoints, and trained artifacts across Colab sessions.